## Stage 2 Unlearning - All 4 Methods

Paper: FIUBench (ICLR 2025)  
Goal: Train GA, GD, KL, PO unlearning on Stage 1 checkpoint using exact hyperparameters from Table 7.

| Method | forget_loss | lr   | batch | accum | epochs | split   |
|--------|-------------|------|-------|-------|--------|---------|
| GA     | ga          | 2e-5 | 8     | 16    | 8      | forget5 |
| GD     | gd          | 2e-5 | 8     | 16    | 8      | forget5 |
| KL     | kl          | 1e-4 | 8     | 16    | 8      | forget5 |
| PO     | idk         | 3e-4 | 8     | 16    | 8      | forget5 |

Run cells sequentially. Each method saves its checkpoint to Drive before the next begins.


## Repo + Drive + GPU




In [12]:
import os
from pathlib import Path

# Clone repo
if not os.path.exists('/content/FIUBench_Reproducing'):
    print("Cloning repo...")
    os.system('git clone https://github.com/akashyall34/FIUBench_Reproducing.git /content/FIUBench_Reproducing')
else:
    # Force-sync to origin/main — discards local patches from previous runs
    print("Repo present — force-syncing to origin/main...")
    os.system('git -C /content/FIUBench_Reproducing fetch origin')
    os.system('git -C /content/FIUBench_Reproducing reset --hard origin/main')

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = f'{PROJECT_ROOT}/FIUBench'

assert os.path.exists(FIUBENCH_DIR), f"FIUBench not found at {FIUBENCH_DIR}"
print(f"✅ PROJECT_ROOT: {PROJECT_ROOT}")

# GPU check
import torch
assert torch.cuda.is_available(), "No GPU — switch runtime before continuing"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Repo present — force-syncing to origin/main...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ PROJECT_ROOT: /content/FIUBench_Reproducing
✅ GPU: NVIDIA A100-SXM4-80GB
✅ VRAM: 85.1 GB


## Install dependencies

In [2]:
import subprocess, sys

deps = [
    "torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1",
    "transformers==4.48.0",
    "xtuner==0.2.0",
    "accelerate==0.34.2",
    "datasets==2.21.0",
    "peft==0.13.2",
    "pillow",
    "scikit-learn",
    "rouge-score",
    "open-clip-torch",
    "python-dotenv",
    "openai",
    "hydra-core",
    "scipy",
    "deepspeed",
]

for dep in deps:
    subprocess.run(f"{sys.executable} -m pip install -q {dep}", shell=True)

import transformers
print(f"✅ torch={torch.__version__}  transformers={transformers.__version__}")


✅ torch=2.10.0+cu128  transformers=4.48.0


In [3]:
import sys
import subprocess

# Reinstall torch/torchvision/torchaudio from the matching CUDA wheel set.
# This is notebook-local and avoids changing the repo code used for reproduction.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--force-reinstall",
        "--index-url",
        "https://download.pytorch.org/whl/cu121",
        "torch==2.4.1",
        "torchvision==0.19.1",
        "torchaudio==2.4.1",
    ],
    check=True,
)

print("✅ Reinstalled matching PyTorch wheels")
print("Restart the notebook kernel now, then rerun from the package install cell onward.")

✅ Reinstalled matching PyTorch wheels
Restart the notebook kernel now, then rerun from the package install cell onward.


## Patches + copy Stage 1 from Drive

In [4]:
import os, re
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
STAGE1_DRIVE = '/content/drive/MyDrive/fiubench_checkpoints/stage1'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'

os.environ['WANDB_MODE']     = 'disabled'
os.environ['HYDRA_FULL_ERROR'] = '1'

# ── 1. Copy Stage 1 from Drive ────────────────────────────────────────────────
if not Path(STAGE1_LOCAL).exists() or not list(Path(STAGE1_LOCAL).glob('*.safetensors')):
    print("Copying Stage 1 from Drive...")
    Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
    ret = os.system(f'rsync -ah --progress {STAGE1_DRIVE}/ {STAGE1_LOCAL}/')
    assert ret == 0, "rsync failed"
safetensors = list(Path(STAGE1_LOCAL).glob('*.safetensors'))
assert safetensors, f"No safetensors in {STAGE1_LOCAL}"
print(f"✅ Stage 1 ready: {[f.name for f in safetensors]}")

# ── 2. Patch forget.py ────────────────────────────────────────────────────────
fg_py = Path(FIUBENCH_DIR) / 'forget.py'
src = fg_py.read_text()
patched = src
patched = patched.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
patched = patched.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')
patched = patched.replace('.to(torch.float16)', '')
patched = patched.replace(
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        **accelerator_log_kwargs)',
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)'
)
patched = patched.replace(
    'accelerator.end_training()\n    output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()',
    'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()'
)
if patched != src:
    fg_py.write_text(patched)
    print("✅ Patched forget.py")
else:
    print("✅ forget.py already patched")

assert 'torch_dtype=torch.bfloat16' in fg_py.read_text(), "bfloat16 patch missing"
assert 'mixed_precision="bf16"'      in fg_py.read_text(), "mixed_precision patch missing"

# ── 3. Patch modeling_llava.py ────────────────────────────────────────────────
import glob
llava_candidates = glob.glob(
    '/usr/local/lib/python*/dist-packages/transformers/models/llava/modeling_llava.py'
)
if llava_candidates:
    llava_path = llava_candidates[0]
    llava_src  = Path(llava_path).read_text()
    llava_patched = re.sub(
        r'n_image_tokens != n_image_features',
        'False',
        llava_src
    )
    if llava_patched != llava_src:
        Path(llava_path).write_text(llava_patched)
        print(f"✅ Patched modeling_llava.py")
    else:
        print("✅ modeling_llava.py already patched")
else:
    print("⚠️  modeling_llava.py not found — may not be needed for this transformers version")

print("\n✅ All patches applied. Ready for Stage 2.")


Copying Stage 1 from Drive...
✅ Stage 1 ready: ['model-00002-of-00002.safetensors', 'model-00001-of-00002.safetensors']
✅ Patched forget.py
✅ Patched modeling_llava.py

✅ All patches applied. Ready for Stage 2.


## Verify Stage 1 Knowledge Injection

In [5]:
# Restore SFHQ images (Colab /content/ is ephemeral — re-download if missing)
import os, subprocess, zipfile
from pathlib import Path

HF_TOKEN       = "" # --> ENTER HF TOKEN
PROJECT_ROOT   = '/content/FIUBench_Reproducing'
sfhq_extracted = Path(PROJECT_ROOT) / 'data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ'
sfhq_symlink   = Path(PROJECT_ROOT) / 'FIUBench/dataset/SFHQ'
zip_dest       = Path('/tmp/SFHQ.zip')
HF_URL         = 'https://huggingface.co/datasets/gray311/FIUBench/resolve/main/SFHQ.zip'

existing = list(sfhq_extracted.glob('*.jpg'))
if existing:
    print(f"✅ SFHQ images already present: {len(existing)}")
else:
    sfhq_extracted.mkdir(parents=True, exist_ok=True)
    # -c resumes partial downloads; --tries retries on dropped connections
    print("Downloading SFHQ.zip (resume-capable, up to 10 retries)...")
    ret = subprocess.run([
        'wget', '-c', '--tries=10', '--timeout=60',
        '--header', f'Authorization: Bearer {HF_TOKEN}',
        '-O', str(zip_dest),
        HF_URL,
    ]).returncode
    assert ret == 0 and zip_dest.exists(), f"wget failed (exit {ret})"
    print(f"Download complete: {zip_dest.stat().st_size / 1e6:.1f} MB")

    print("Extracting ...")
    with zipfile.ZipFile(zip_dest, 'r') as zf:
        for member in zf.infolist():
            if member.filename.endswith('.jpg'):
                member.filename = Path(member.filename).name
                zf.extract(member, sfhq_extracted)
    zip_dest.unlink()  # free space
    print(f"✅ Extracted {len(list(sfhq_extracted.glob('*.jpg')))} images")

# Ensure symlink so FIUBench/dataset/SFHQ resolves correctly
if not sfhq_symlink.exists():
    sfhq_symlink.parent.mkdir(parents=True, exist_ok=True)
    sfhq_symlink.symlink_to(sfhq_extracted)
    print(f"✅ Symlink: {sfhq_symlink} → {sfhq_extracted}")
else:
    print(f"✅ {sfhq_symlink} already exists")


Download complete: 146.0 MB
Extracting ...
✅ Extracted 1000 images
✅ Symlink: /content/FIUBench_Reproducing/FIUBench/dataset/SFHQ → /content/FIUBench_Reproducing/data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ


In [ ]:
import json, torch
from pathlib import Path
from transformers import AutoProcessor, LlavaForConditionalGeneration
from PIL import Image

STAGE1_LOCAL  = '/content/stage1_final'
PROJECT_ROOT  = '/content/FIUBench_Reproducing'
DATA_PATH     = f'{PROJECT_ROOT}/FIUBench/dataset/full.json'
SFHQ_DIR      = f'{PROJECT_ROOT}/FIUBench/dataset/SFHQ'
ALT_SFHQ_DIR  = f'{PROJECT_ROOT}/data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ'

print("Loading Stage 1 model for verification...")
processor = AutoProcessor.from_pretrained(STAGE1_LOCAL, trust_remote_code=True)
model = LlavaForConditionalGeneration.from_pretrained(
    STAGE1_LOCAL, torch_dtype=torch.bfloat16, trust_remote_code=True
).cuda().eval()

# Load forget5 identities
with open(DATA_PATH) as f:
    full_data = [json.loads(line) for line in f if line.strip()]

# Resolve forget5 split  (file is split.json, not splits.json)
splits_path = Path(PROJECT_ROOT) / 'FIUBench/dataset/split.json'
if splits_path.exists():
    with open(splits_path) as f:
        splits = json.load(f)
    forget5_ids = set(splits['forget5'])
else:
    # fallback: first 20 unique IDs
    all_ids = []
    for item in full_data:
        uid = item.get('unique_id', item.get('image_path', ''))
        if uid not in all_ids:
            all_ids.append(uid)
    forget5_ids = set(all_ids[:20])

forget5_items = [x for x in full_data if x.get('unique_id', x.get('image_path','')) in forget5_ids]
print(f"Forget5 identities: {len(set(x.get('unique_id','') for x in forget5_items))}")

# Run on first 5 QA pairs
from rouge_score import rouge_scorer as rs_module
scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)

rouge_scores = []
for item in forget5_items[:5]:
    # Resolve image path robustly across FIUBench layouts
    raw_img_path = item.get('image_path', '')
    img_name = Path(raw_img_path).name
    normalized_rel = raw_img_path.replace('./', '')
    candidates = [
        raw_img_path,
        normalized_rel,
        f"{PROJECT_ROOT}/FIUBench/{normalized_rel}",
        f"{PROJECT_ROOT}/FIUBench/dataset/{normalized_rel}",
        f"{SFHQ_DIR}/{img_name}",
        f"{ALT_SFHQ_DIR}/{img_name}",
    ]
    img_path = None
    for candidate in candidates:
        if Path(candidate).exists():
            img_path = candidate
            break

    if img_path is None:
        print(f"  Skipped (image not found: {raw_img_path})\n")
        continue

    qa_entries = item.get('qa_list', item.get('QA list', []))
    if not qa_entries:
        print("  Skipped (no QA entries found)\n")
        continue

    qa = qa_entries[0]
    question = qa.get('question', '')
    answer = qa.get('answer', '').lower()
    if not question or not answer:
        print("  Skipped (missing question/answer)\n")
        continue

    try:
        image = Image.open(img_path).convert('RGB')
        prompt = f"USER: <image>\n{question}\nASSISTANT:"
        inputs = processor(text=prompt, images=image, return_tensors='pt').to('cuda')
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        pred = processor.decode(out[0], skip_special_tokens=True)
        pred = pred.split('ASSISTANT:')[-1].strip().lower()
        score = scorer.score(answer, pred)['rougeL'].fmeasure
        rouge_scores.append(score)
        print(f"  Q: {question[:60]}")
        print(f"  GT:   {answer[:80]}")
        print(f"  Pred: {pred[:80]}")
        print(f"  ROUGE-L: {score:.3f}\n")
    except Exception as e:
        print(f"  Skipped (error: {e})\n")

if rouge_scores:
    mean_rouge = sum(rouge_scores) / len(rouge_scores)
    print(f"Mean ROUGE-L on {len(rouge_scores)} forget5 samples: {mean_rouge:.3f}")
    if mean_rouge > 0.5:
        print("✅ Knowledge injection confirmed — ROUGE-L >> pretrained baseline (~0.034)")
    else:
        print("⚠️  Low ROUGE — Stage 1 may not have converged. Check checkpoint before proceeding.")
else:
    print("⚠️  No samples evaluated — check image paths")

del model
torch.cuda.empty_cache()

## Stage 2: Gradient Ascent (GA)

In [ ]:
import os, subprocess, time
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
METHOD       = 'ga'
LR           = '2e-5'
STAGE2_LOCAL = f'/content/stage2_{METHOD}'
STAGE2_DRIVE = f'/content/drive/MyDrive/fiubench_checkpoints/stage2_forget5/{METHOD}'

Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29510 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/forget_{METHOD}_log.txt"
)

print(f"Starting Stage 2 [{METHOD.upper()}]  lr={LR}  batch=8  accum=16  epochs=8  split=forget5")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    print(f"✅ GA checkpoints saved to {STAGE2_DRIVE}")
else:
    print(f"❌ GA failed (exit {ret}). Check /tmp/forget_{METHOD}_log.txt")


## Stage 2: Gradient Difference (GD)

In [ ]:
import os, subprocess, time
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
METHOD       = 'gd'
LR           = '2e-5'
STAGE2_LOCAL = f'/content/stage2_{METHOD}'
STAGE2_DRIVE = f'/content/drive/MyDrive/fiubench_checkpoints/stage2_forget5/{METHOD}'

Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29511 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/forget_{METHOD}_log.txt"
)

print(f"Starting Stage 2 [{METHOD.upper()}]  lr={LR}  batch=8  accum=16  epochs=8  split=forget5")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    print(f"✅ GD checkpoints saved to {STAGE2_DRIVE}")
else:
    print(f"❌ GD failed (exit {ret}). Check /tmp/forget_{METHOD}_log.txt")


## Stage 2: KL Minimization (KL)

In [ ]:
import os, subprocess, time
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
METHOD       = 'kl'
LR           = '1e-4'
STAGE2_LOCAL = f'/content/stage2_{METHOD}'
STAGE2_DRIVE = f'/content/drive/MyDrive/fiubench_checkpoints/stage2_forget5/{METHOD}'

Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29512 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/forget_{METHOD}_log.txt"
)

print(f"Starting Stage 2 [{METHOD.upper()}]  lr={LR}  batch=8  accum=16  epochs=8  split=forget5")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    print(f"✅ KL checkpoints saved to {STAGE2_DRIVE}")
else:
    print(f"❌ KL failed (exit {ret}). Check /tmp/forget_{METHOD}_log.txt")


## Stage 2: Preference Optimization / IDK (PO)

In [ ]:
import os, subprocess, time
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final'
FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
METHOD       = 'idk'
LABEL        = 'po'
LR           = '3e-4'
STAGE2_LOCAL = f'/content/stage2_{LABEL}'
STAGE2_DRIVE = f'/content/drive/MyDrive/fiubench_checkpoints/stage2_forget5/{LABEL}'

Path(STAGE2_LOCAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29513 forget.py "
    f"--config-name forget_lora "
    f"model_path={STAGE1_LOCAL} "
    f"save_dir={STAGE2_LOCAL} "
    f"split=forget5 "
    f"forget_loss={METHOD} "
    f"lr={LR} "
    f"batch_size=8 "
    f"num_epochs=8 "
    f"seed=233 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/forget_{LABEL}_log.txt"
)

print(f"Starting Stage 2 [PO/IDK]  lr={LR}  batch=8  accum=16  epochs=8  split=forget5")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(STAGE2_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {STAGE2_LOCAL}/ {STAGE2_DRIVE}/")
    print(f"✅ PO checkpoints saved to {STAGE2_DRIVE}")
else:
    print(f"❌ PO failed (exit {ret}). Check /tmp/forget_{LABEL}_log.txt")


## Retain model baseline

In [ ]:
# The retain model is fine-tuned on S_R only (never sees forget5 identities).
# It is the ideal unlearning upper bound used for KS-Test in Day 5.
# Training: same as Stage 1 but data filtered to retain95 identities only.

import os, subprocess, time, json
from pathlib import Path

PROJECT_ROOT   = '/content/FIUBench_Reproducing'
FIUBENCH_DIR   = f'{PROJECT_ROOT}/FIUBench'
PRETRAINED_ID  = 'xtuner/llava-phi-3-mini-hf'   # start from pretrained, not Stage 1
RETAIN_LOCAL   = '/content/retain_model'
RETAIN_DRIVE   = '/content/drive/MyDrive/fiubench_checkpoints/retain_model'
DATA_PATH      = f'{FIUBENCH_DIR}/dataset/full.json'
RETAIN_DATA    = '/tmp/retain95.json'

Path(RETAIN_LOCAL).mkdir(parents=True, exist_ok=True)

# Build retain95 dataset (identities NOT in forget5)
with open(DATA_PATH) as f:
    full_data = [json.loads(line) for line in f if line.strip()]

splits_path = Path(PROJECT_ROOT) / 'FIUBench/dataset/split.json'  # file is split.json, not splits.json
if splits_path.exists():
    with open(splits_path) as f:
        splits = json.load(f)
    forget5_ids = set(splits['forget5'])
else:
    all_ids = list({x.get('unique_id', x.get('image_path','')) for x in full_data})
    forget5_ids = set(all_ids[:20])

retain95_data = [x for x in full_data
                 if x.get('unique_id', x.get('image_path','')) not in forget5_ids]
print(f"Retain95 identities: {len({x.get('unique_id','') for x in retain95_data})}")
print(f"Retain95 QA pairs:   {len(retain95_data)}")

with open(RETAIN_DATA, 'w') as f:
    json.dump(retain95_data, f)
print(f"✅ Saved retain95 dataset to {RETAIN_DATA}")

# Fine-tune on retain95 using same Stage 1 hyperparameters
# lr=2e-5, batch=8, accum=16, epochs=10, full fine-tuning (no LoRA)
cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29514 finetune.py "
    f"--config-name finetune_llava_phi "
    f"save_dir={RETAIN_LOCAL} "
    f"data_path={RETAIN_DATA} "
    f"split=full "
    f"batch_size=8 "
    f"gradient_accumulation_steps=16 "
    f"overwrite_dir=true "
    f"2>&1 | tee /tmp/retain_model_log.txt"
)
# Note: model_id, lr=2e-5, num_epochs=10, seed=0 are correct yaml defaults — no override needed

print("\nStarting retain model fine-tuning on S_R only...")
print("=" * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
ret = proc.returncode

elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {ret}  |  Time: {h}h {m}m {s}s")

if ret == 0:
    Path(RETAIN_DRIVE).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah --progress {RETAIN_LOCAL}/ {RETAIN_DRIVE}/")
    print(f"✅ Retain model saved to {RETAIN_DRIVE}")
else:
    print(f"❌ Retain model training failed. Check /tmp/retain_model_log.txt")


## Stage 2 Evaluation - FIUBench Metrics (forget5)

Runs evaluate_util.py on each checkpoint (retain + GA/GD/KL/PO), then aggregates with aggregate_eval_stat.py.

Output per method: Forget Quality (KS-Test p-value), Model Utility (ROUGE/Prob/Truth Ratio harmonic mean).

In [6]:
import os, subprocess, shutil
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = f'{PROJECT_ROOT}/FIUBench'
STAGE1_LOCAL = '/content/stage1_final'
DRIVE_ROOT   = '/content/drive/MyDrive/fiubench_checkpoints'
RETAIN_LOCAL = '/content/retain_model'
RETAIN_DRIVE = f'{DRIVE_ROOT}/retain_model'

METHODS = {
    'ga': {'local': '/content/stage2_ga', 'drive': f'{DRIVE_ROOT}/stage2_forget5/ga'},
    'gd': {'local': '/content/stage2_gd', 'drive': f'{DRIVE_ROOT}/stage2_forget5/gd'},
    'kl': {'local': '/content/stage2_kl', 'drive': f'{DRIVE_ROOT}/stage2_forget5/kl'},
    'po': {'local': '/content/stage2_po', 'drive': f'{DRIVE_ROOT}/stage2_forget5/po'},
}

# -- Patch evaluate_util.py ----------------------------------------------------
_eu = Path(f"{FIUBENCH_DIR}/evaluate_util.py")
_eu_src = _eu.read_text()
_eu_new = _eu_src
_eu_new = _eu_new.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
_eu_new = _eu_new.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')
_eu_new = _eu_new.replace('model.half().cuda()', 'model.cuda()')
if _eu_new != _eu_src:
    _eu.write_text(_eu_new)
    print('Patched evaluate_util.py: flash_attention_2->sdpa, float16->bfloat16, .half() removed')
else:
    print('evaluate_util.py already patched')

# -- Restore Stage 1 from Drive -------------------------------------------------
if not Path(STAGE1_LOCAL).exists() or not list(Path(STAGE1_LOCAL).glob('*.safetensors')):
    print('Restoring stage1 from Drive...')
    Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah {DRIVE_ROOT}/stage1/ {STAGE1_LOCAL}/")
else:
    print(f'Stage1 at {STAGE1_LOCAL}')

# -- Restore retain model from Drive --------------------------------------------
if not Path(RETAIN_LOCAL).exists() or not list(Path(RETAIN_LOCAL).glob('*.safetensors')):
    print('Restoring retain model from Drive...')
    Path(RETAIN_LOCAL).mkdir(parents=True, exist_ok=True)
    os.system(f"rsync -ah {RETAIN_DRIVE}/ {RETAIN_LOCAL}/")
else:
    print(f'Retain model at {RETAIN_LOCAL}')

# -- Restore Stage 2 checkpoints from Drive -------------------------------------
for method, paths in METHODS.items():
    ckpt = Path(paths['local']) / 'checkpoint.pt'
    if not ckpt.exists():
        print(f"Restoring {method} from Drive...")
        Path(paths['local']).mkdir(parents=True, exist_ok=True)
        os.system(f"rsync -ah {paths['drive']}/ {paths['local']}/")
    else:
        print(f"{method} checkpoint at {ckpt}")

Patched evaluate_util.py: flash_attention_2->sdpa, float16->bfloat16, .half() removed
Stage1 at /content/stage1_final
Restoring retain model from Drive...
Restoring ga from Drive...
Restoring gd from Drive...
Restoring kl from Drive...
Restoring po from Drive...


In [13]:
# Evaluate retain model on forget5 + retain5 splits.
# This produces the reference distribution for KS-Test (Forget Quality).
import os, subprocess, time
from pathlib import Path

FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
RETAIN_LOCAL = '/content/retain_model'
RETAIN_EVAL  = f'{RETAIN_LOCAL}/eval_results'
Path(RETAIN_EVAL).mkdir(parents=True, exist_ok=True)

cmd = (
    f"bash -c 'set -o pipefail; cd {FIUBENCH_DIR} && "
    f"python evaluate_util.py --config-name eval "
    f"model_path={RETAIN_LOCAL} "
    f"save_dir={RETAIN_EVAL} "
    f"'LoRA.r=0' "
    f"'hydra.run.dir=/tmp/hydra_eval_retain' "
    f"2>&1 | tee /tmp/eval_retain_log.txt'"
)

print('Evaluating retain model (forget5 + retain5)...')
print('=' * 70)
t0 = time.time()
proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
elapsed = time.time() - t0
h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
print(f"\nExit: {proc.returncode}  |  Time: {h}h {m}m {s}s")

agg = Path(RETAIN_EVAL) / 'retain5_eval_log_aggregated.json'
print(f"{'PASS' if agg.exists() else 'FAIL'} Retain eval log: {agg}")

Evaluating retain model (forget5 + retain5)...
2026-04-15 15:40:13.231031: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 15:40:13.259476: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776267613.282336   12782 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776267613.290222   12782 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776267613.314230   12782 computation_placer.cc:177] computation placer already regi

In [ ]:
# Evaluate each unlearning method on forget5 + retain5.
# LoRA weights loaded on top of Stage 1 model; save_dir auto-set to {method_dir}/eval_results/.
import os, subprocess, time
from pathlib import Path

FIUBENCH_DIR = '/content/FIUBench_Reproducing/FIUBench'
STAGE1_LOCAL = '/content/stage1_final'

METHODS = {
    'ga': '/content/stage2_ga',
    'gd': '/content/stage2_gd',
    'kl': '/content/stage2_kl',
    'po': '/content/stage2_po',
}

for method, method_dir in METHODS.items():
    ckpt = f"{method_dir}/checkpoint.pt"
    if not Path(ckpt).exists():
        print(f"Skipping {method}: {ckpt} not found")
        continue

    print(f"\n{'='*70}")
    print(f"Evaluating {method.upper()} (forget5 + retain5)...")
    print(f"{'='*70}")
    t0 = time.time()

    cmd = (
        f"bash -c 'set -o pipefail; cd {FIUBENCH_DIR} && "
        f"python evaluate_util.py --config-name eval "
        f"model_path={STAGE1_LOCAL} "
        f"'LoRA.r=128' "
        f"'LoRA.alpha=256' "
        f"'LoRA.lora_path={ckpt}' "
        f"'hydra.run.dir=/tmp/hydra_eval_{method}' "
        f"2>&1 | tee /tmp/eval_{method}_log.txt'"
    )

    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    elapsed = time.time() - t0
    h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
    print(f"\nExit: {proc.returncode}  |  Time: {h}h {m}m {s}s")

    agg = Path(method_dir) / 'eval_results' / 'retain5_eval_log_aggregated.json'
    print(f"{'PASS' if agg.exists() else 'FAIL'} Eval log: {agg}")

In [ ]:
# Aggregate results: compute Forget Quality (KS-Test) + Model Utility for each method.
import os, subprocess, json, csv
from pathlib import Path

FIUBENCH_DIR   = '/content/FIUBench_Reproducing/FIUBench'
RETAIN_AGG     = '/content/retain_model/eval_results/retain5_eval_log_aggregated.json'
RESULTS_DIR    = '/content/eval_results'
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

METHODS = {
    'GA': '/content/stage2_ga/eval_results/retain5_eval_log_aggregated.json',
    'GD': '/content/stage2_gd/eval_results/retain5_eval_log_aggregated.json',
    'KL': '/content/stage2_kl/eval_results/retain5_eval_log_aggregated.json',
    'PO': '/content/stage2_po/eval_results/retain5_eval_log_aggregated.json',
}

if not Path(RETAIN_AGG).exists():
    print(f"Retain eval log not found: {RETAIN_AGG}")
    print('Run the retain eval cell first.')
else:
    all_results = []
    for method, ckpt_agg in METHODS.items():
        if not Path(ckpt_agg).exists():
            print(f"Skipping {method}: {ckpt_agg} not found")
            continue
        save_csv = f"{RESULTS_DIR}/{method.lower()}_aggr.csv"
        cmd = (
            f"bash -c 'set -o pipefail; cd {FIUBENCH_DIR} && "
            f"python aggregate_eval_stat.py "
            f"retain_result={RETAIN_AGG} "
            f"ckpt_result={ckpt_agg} "
            f"method_name={method} "
            f"save_file={save_csv} "
            f"'hydra.run.dir=/tmp/hydra_agg_{method.lower()}' "
            f"2>&1'"
        )
        ret = os.system(cmd)
        if ret == 0 and Path(save_csv).exists():
            with open(save_csv) as f:
                rows = list(csv.DictReader(f))
                if rows:
                    all_results.append(rows[0])

    if all_results:
        print('\n' + '='*80)
        print('FIUBench Results (forget5)')
        print('='*80)
        keys = ['Method', 'Forget Quality', 'KS Test PVal Forget', 'Model Utility',
                'ROUGE Forget', 'ROUGE Retain', 'Prob. Forget', 'Prob. Retain',
                'Truth Ratio Forget', 'Truth Ratio Retain']
        header = f"{'Method':<8}" + ''.join(f"  {k:<22}" for k in keys[1:])
        print(header)
        print('-' * len(header))
        for r in all_results:
            row = f"{r.get('Method',''):<8}"
            for k in keys[1:]:
                v = r.get(k, 'N/A')
                try:
                    row += f"  {float(v):<22.4f}"
                except Exception:
                    row += f"  {str(v):<22}"
            print(row)
        print('='*80)
        DRIVE_RESULTS = '/content/drive/MyDrive/fiubench_checkpoints/eval_results_forget5'
        os.system(f"rsync -ah {RESULTS_DIR}/ {DRIVE_RESULTS}/")
        print(f"\nResults synced to Drive: {DRIVE_RESULTS}")